In [ ]:
from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
import torch

# 1. Check if GPU is available and set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Load and validate the dataset
dataset = load_dataset("csv", data_files="./../Dataset/data1.csv")

# Validate dataset fields
def validate_dataset(dataset):
    required_fields = ['question', 'context', 'answer']
    for field in required_fields:
        if field not in dataset.column_names:
            raise ValueError(f"Missing required field: {field}")
    print("Dataset validation passed!")

validate_dataset(dataset["train"])

# Remove rows with missing or null values
train_dataset = dataset["train"].filter(
    lambda example: example['question'] is not None and example['context'] is not None and example['answer'] is not None
)

# 3. Split dataset into training and evaluation sets
split_dataset = train_dataset.train_test_split(test_size=0.2)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

# 4. Load the T5 model and tokenizer
model_name = "t5-large"  # Use "t5-large" or "t5-small" for a smaller model
model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)  # Move model to selected device
tokenizer = T5Tokenizer.from_pretrained(model_name)

# 5. Preprocess the dataset
def preprocess_function(examples):
    # Combine question and context into a single string for T5
    inputs = [f"question: {q} context: {c}" for q, c in zip(examples['question'], examples['context'])]
    targets = examples['answer']
    
    # Ensure inputs and targets are strings
    inputs = [str(input_text) for input_text in inputs]
    targets = [str(target_text) for target_text in targets]
    
    # Tokenize inputs and targets
    model_inputs = tokenizer(inputs, max_length=512, padding="max_length", truncation=True)
    labels = tokenizer(targets, max_length=128, padding="max_length", truncation=True)

    # Assign tokenized labels
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Tokenize the datasets
tokenized_train_dataset = train_dataset.map(preprocess_function, batched=True)
tokenized_eval_dataset = eval_dataset.map(preprocess_function, batched=True)

# 6. Set up training arguments with GPU optimizations
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,
    save_steps=10_000,
    save_total_limit=2,
    logging_dir='./logs',
    logging_steps=500,
    evaluation_strategy="steps",  # Enable evaluation during training
    eval_steps=10_000,  # Match save_steps for compatibility
    load_best_model_at_end=True,  # Load best model after evaluation
    fp16=True,  # Use mixed precision for faster training on GPUs
)

# 7. Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    tokenizer=tokenizer,
)

# 8. Train the model
trainer.train()

# 9. Save the fine-tuned model
model.save_pretrained('./fine_tuned_t5_large')
tokenizer.save_pretrained('./fine_tuned_t5_large')

# 10. Optionally, evaluate the model
results = trainer.evaluate(eval_dataset=tokenized_eval_dataset)

# Print evaluation results
print("Evaluation Results:", results)


Using device: cuda
Dataset validation passed!


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Map:   0%|          | 0/178 [00:00<?, ? examples/s]

Map:   0%|          | 0/45 [00:00<?, ? examples/s]

C:\Users\kavin\AppData\Roaming\Python\Python312\site-packages\transformers\training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
C:\Users\kavin\AppData\Local\Temp\ipykernel_20040\1011773019.py:76: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


  0%|          | 0/15 [00:00<?, ?it/s]

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch

# Load the fine-tuned model and tokenizer
model_path = './fine_tuned_t5'
model = T5ForConditionalGeneration.from_pretrained(model_path)
tokenizer = T5Tokenizer.from_pretrained(model_path)

# Check if a GPU is available and move model to GPU if so
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Define the question and context
question = "what supervised learninng?"
context = "Supervised learning is a type of machine learning where the model is trained using labeled data. The goal is to teach the model to make predictions based on known outcomes."
# Format input
input_text = f"question: {question} context: {context}"

# Tokenize the input
input_ids = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).input_ids

# Move input tensors to the same device as the model (GPU or CPU)
input_ids = input_ids.to(device)

# Generate the output
output_ids = model.generate(input_ids, max_length=250, num_beams=5, early_stopping=True)

# Decode the output to text
answer = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print(f"Answer: {answer}")